# Why Do Models Fail on Rare Diseases?
## Notebook 2 — Shared Backbone Transformer (CCS label space, MIMIC-IV)

Architecture exactly matching MedCo (model.py):
  Step 1  Three separate embedding matrices Z_dx, Z_rx, Z_px ∈ R^{C_i × d}
  Step 2  v_t^(h) = SUM pooling of codes per modality per visit (NOT mean)
  Step 3  Three separate Transformer encoders (one per modality, shared weights)
  Step 4  Concatenate → h_p ∈ R^{3d} → FC → sigmoid → x ∈ [0,1]^N
  Eval    AUPRC (primary, threshold-free) 

> **Note added during the November 2026 review (code unchanged).** Training stopped after epoch 7 (no early-stopping message; epoch 7 took 967 min on the laptop). The test evaluation uses the best validation state, epoch 5 (val sample-AUPRC 0.4723). The headline "MedCo-style" cell reports sample-AUPRC restricted to each label-frequency quartile, with quartiles defined on *sample-level* training counts (Q25 = 200), which differ from the patient-level quartiles stored in `meta_ccs.json` (Q25 = 166). The later cell "10. Test Evaluation" was not executed.

## 0. Installs

In [1]:
%pip install pyhealth torch scikit-learn numpy tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and Paths

In [2]:
import json
import pickle
import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, average_precision_score
from tqdm import tqdm

ROOT        = Path('..')
PROCESSED   = ROOT / 'processed'
RESULTS     = ROOT / 'results'
CHECKPOINTS = ROOT / 'checkpoints'
for d in [RESULTS, CHECKPOINTS]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else
    'cpu'
)
print(f"Device: {DEVICE}")

Device: mps


## 2. Load Notebook 1 Artifacts

In [3]:
with open(PROCESSED / 'meta_ccs.json') as f:
    meta = json.load(f)

code2idx   = meta['code2idx']   # CCS code → target index
D_out      = meta['N']          # number of CCS diagnosis codes to predict
rare_idx   = meta['rare_idx']
medium_idx = meta['medium_idx']
common_idx = meta['common_idx']
m_median   = meta['m_median']
m_90       = meta['m_90']

def load_pkl(name):
    with open(PROCESSED / name, 'rb') as f:
        return pickle.load(f)

train_samples     = load_pkl('train_samples.pkl')
val_samples       = load_pkl('val_samples.pkl')
test_samples      = load_pkl('test_samples.pkl')
patient_sequences = load_pkl('patient_sequences.pkl')

print(f"CCS target vocabulary N = {D_out}")
print(f"Train / Val / Test      = "
      f"{len(train_samples):,} / {len(val_samples):,} / {len(test_samples):,}")
print(f"Patient sequences       = {len(patient_sequences):,} patients")

CCS target vocabulary N = 572
Train / Val / Test      = 225,024 / 15,004 / 15,010
Patient sequences       = 100,163 patients


## 3. Build Three Separate Input Vocabularies (conditions / drugs / procedures)

MedCo keeps three SEPARATE embedding matrices:
  self.embeddings3['conditions'] — ICD-9 Dx codes
  self.embeddings3['drugs']      — ATC level-3 Rx codes
  self.embeddings3['procedures'] — ICD-9 Px codes
Each has its own vocabulary and embedding matrix.
Index 0 is reserved for padding in each.

In [4]:
dx_vocab_set, rx_vocab_set, px_vocab_set = set(), set(), set()
for pid, visits in patient_sequences.items():
    for _, dx, px, rx in visits:
        dx_vocab_set.update(dx)
        px_vocab_set.update(px)
        rx_vocab_set.update(rx)

dx_vocab = sorted(dx_vocab_set)
rx_vocab = sorted(rx_vocab_set)
px_vocab = sorted(px_vocab_set)

# PAD index = 0, real codes start at 1
dx2idx = {c: i + 1 for i, c in enumerate(dx_vocab)}
rx2idx = {c: i + 1 for i, c in enumerate(rx_vocab)}
px2idx = {c: i + 1 for i, c in enumerate(px_vocab)}

C_dx = len(dx_vocab) + 1
C_rx = len(rx_vocab) + 1
C_px = len(px_vocab) + 1

print(f"Vocabulary sizes:")
print(f"  conditions (ICD-9 Dx)  : {C_dx - 1:,} codes  → C_dx = {C_dx}")
print(f"  drugs      (ATC Rx)    : {C_rx - 1:,} codes  → C_rx = {C_rx}")
print(f"  procedures (ICD-9 Px)  : {C_px - 1:,} codes  → C_px = {C_px}")

Vocabulary sizes:
  conditions (ICD-9 Dx)  : 576 codes  → C_dx = 577
  drugs      (ATC Rx)    : 201 codes  → C_rx = 202
  procedures (ICD-9 Px)  : 230 codes  → C_px = 231


## 4. CCS History Mapping (for visit masking)
We need the CCS history per sample to mask already-diagnosed codes.

In [5]:
def encode_visit_split(dx, px, rx):
    """
    Returns three lists of integer indices (one per modality).
    Index 0 = PAD (skipped). Empty lists get [0] as fallback.
    """
    dx_enc = [dx2idx[c] for c in dx if c in dx2idx] or [0]
    rx_enc = [rx2idx[c] for c in rx if c in rx2idx] or [0]
    px_enc = [px2idx[c] for c in px if c in px2idx] or [0]
    return dx_enc, rx_enc, px_enc

def build_sequence(sample):
    """
    Returns:
        dx_seq : list of lists (one per visit, ICD-9 Dx indices)
        rx_seq : list of lists (one per visit, ATC Rx indices)
        px_seq : list of lists (one per visit, ICD-9 Px indices)
        hist_idx : set of CCS target indices already seen by visit k
    """
    pid, k = sample['patient_id'], sample['k']
    visits = patient_sequences[pid][:k]   # visits 1..k (chronological)

    dx_seq, rx_seq, px_seq = [], [], []
    history_ccs = set()

    for _, dx, px, rx in visits:
        dx_enc, rx_enc, px_enc = encode_visit_split(dx, px, rx)
        dx_seq.append(dx_enc)
        rx_seq.append(rx_enc)
        px_seq.append(px_enc)
        history_ccs.update(dx)

    hist_idx = {code2idx[c] for c in history_ccs if c in code2idx}
    return dx_seq, rx_seq, px_seq, hist_idx

## 5. Dataset and Collate

MedCo uses batch_encode_3d: shape (B, V, C) where V=visits, C=codes per visit.
We replicate this with three separate tensors (one per modality).

In [6]:
# ============================================================
# Cell 5 — Collate, Pre-compute, DataLoaders
# ============================================================

BATCH_SIZE = 64

def pad_modality(seqs_batch, pad_idx=0):
    B = len(seqs_batch)
    T = max(len(s) for s in seqs_batch)
    L = max((len(v) for s in seqs_batch for v in s), default=1)
    code_idx  = torch.zeros(B, T, L, dtype=torch.long)
    code_mask = torch.zeros(B, T, L, dtype=torch.float)
    pad_mask  = torch.ones(B, T, dtype=torch.bool)
    lengths   = torch.zeros(B, dtype=torch.long)
    for b, seq in enumerate(seqs_batch):
        lengths[b] = len(seq) - 1
        for t, visit in enumerate(seq):
            pad_mask[b, t] = False
            for l, code in enumerate(visit):
                code_idx[b, t, l] = code
                if code != pad_idx:
                    code_mask[b, t, l] = 1.0
    return code_idx, code_mask, pad_mask, lengths

def collate_fn(batch):
    dx_seqs, rx_seqs, px_seqs, ys, hist_masks = zip(*batch)
    dx_idx, dx_mask, dx_pad, dx_len = pad_modality(dx_seqs)
    rx_idx, rx_mask, rx_pad, rx_len = pad_modality(rx_seqs)
    px_idx, px_mask, px_pad, px_len = pad_modality(px_seqs)
    y    = torch.stack(ys)
    hist = torch.stack(hist_masks)
    return (dx_idx, dx_mask, dx_pad, dx_len,
            rx_idx, rx_mask, rx_pad, rx_len,
            px_idx, px_mask, px_pad, px_len,
            y, hist)

def precompute_sequences(samples, patient_sequences, code2idx,
                          dx2idx, rx2idx, px2idx, D_out):
    processed = []
    for s in tqdm(samples, desc="Pre-computing"):
        pid, k  = s['patient_id'], s['k']
        visits  = patient_sequences[pid][:k]
        dx_seq, rx_seq, px_seq = [], [], []
        history_ccs = set()
        for _, dx, px, rx in visits:
            dx_seq.append([dx2idx[c] for c in dx if c in dx2idx] or [0])
            rx_seq.append([rx2idx[c] for c in rx if c in rx2idx] or [0])
            px_seq.append([px2idx[c] for c in px if c in px2idx] or [0])
            history_ccs.update(dx)
        y = torch.zeros(D_out)
        for c in s['label']:
            if c in code2idx:
                y[code2idx[c]] = 1.0
        hist_mask = torch.zeros(D_out, dtype=torch.bool)
        for c in history_ccs:
            if c in code2idx:
                hist_mask[code2idx[c]] = True
        processed.append((dx_seq, rx_seq, px_seq, y, hist_mask))
    return processed

print("Pre-computing train sequences...")
train_proc = precompute_sequences(train_samples, patient_sequences,
                                   code2idx, dx2idx, rx2idx, px2idx, D_out)
print("Pre-computing val sequences...")
val_proc   = precompute_sequences(val_samples, patient_sequences,
                                   code2idx, dx2idx, rx2idx, px2idx, D_out)
print("Pre-computing test sequences...")
test_proc  = precompute_sequences(test_samples, patient_sequences,
                                   code2idx, dx2idx, rx2idx, px2idx, D_out)
print("Done.")

class PrecomputedDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        return self.data[i]

train_loader = DataLoader(PrecomputedDataset(train_proc),
                          batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(PrecomputedDataset(val_proc),
                          batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(PrecomputedDataset(test_proc),
                          batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

Pre-computing train sequences...


Pre-computing: 100%|██████████| 225024/225024 [00:14<00:00, 16047.61it/s]


Pre-computing val sequences...


Pre-computing: 100%|██████████| 15004/15004 [00:00<00:00, 32186.57it/s]


Pre-computing test sequences...


Pre-computing: 100%|██████████| 15010/15010 [00:00<00:00, 32696.13it/s]

Done.
Train batches : 3,516
Val batches   : 235
Test batches  : 235


## 6. Backbone Model — three transformers (diagnoses Dx, medications Rx, procedures Px)

Key differences from naive implementation:
  - Three SEPARATE embedding matrices (not one shared Z)
  - SUM pooling within visit (line 653 of model.py: torch.sum(x, dim=2))
  - Three SEPARATE Transformers (one per modality)
  - Concatenation of patient embeddings → 3d → FC → N

In [7]:
from pyhealth.models import TransformerLayer

class SharedBackbone(nn.Module):
    def __init__(self, C_dx, C_rx, C_px, D_out,
                 d=256, n_heads=1, n_layers=1, dropout=0.5):
        super().__init__()

        # Step 1: separate embedding matrices
        self.emb_dx = nn.Embedding(C_dx, d, padding_idx=0)
        self.emb_rx = nn.Embedding(C_rx, d, padding_idx=0)
        self.emb_px = nn.Embedding(C_px, d, padding_idx=0)

        # Match MedCo's llm_residual_scale=0.01 init
        # (prevents sum-pooling explosion with default std≈1 init)
        for emb in [self.emb_dx, self.emb_rx, self.emb_px]:
            nn.init.normal_(emb.weight, mean=0.0, std=0.01)
            emb.weight.data[0].fill_(0)  # reset PAD token to zero

        # Step 3: separate PyHealth TransformerLayers
        self.transformer_dx = TransformerLayer(
            d, heads=n_heads, dropout=dropout, num_layers=n_layers)
        self.transformer_rx = TransformerLayer(
            d, heads=n_heads, dropout=dropout, num_layers=n_layers)
        self.transformer_px = TransformerLayer(
            d, heads=n_heads, dropout=dropout, num_layers=n_layers)

        # Step 4: FC over concatenated patient embeddings
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(3 * d, D_out)

    def _encode_modality(self, emb_layer, transformer,
                          code_idx, code_mask, pad_mask, lengths):
        emb = emb_layer(code_idx)                        # (B, T, L, d)
        v   = (emb * code_mask.unsqueeze(-1)).sum(dim=2) # (B, T, d)
        visit_mask = (~pad_mask).long()
        _, x = transformer(v, visit_mask)
        return x

    def forward(self, dx_idx, dx_mask, dx_pad, dx_len,
                      rx_idx, rx_mask, rx_pad, rx_len,
                      px_idx, px_mask, px_pad, px_len,
                      hist_mask=None):
        x_dx = self._encode_modality(
            self.emb_dx, self.transformer_dx,
            dx_idx, dx_mask, dx_pad, dx_len)
        x_rx = self._encode_modality(
            self.emb_rx, self.transformer_rx,
            rx_idx, rx_mask, rx_pad, rx_len)
        x_px = self._encode_modality(
            self.emb_px, self.transformer_px,
            px_idx, px_mask, px_pad, px_len)

        patient_emb = torch.cat([x_dx, x_rx, x_px], dim=1)
        patient_emb = self.dropout(patient_emb)
        logits      = self.fc(patient_emb)

        if hist_mask is not None:
            logits = logits.masked_fill(hist_mask, -1e4)
        return logits

# Instantiate
model = SharedBackbone(
    C_dx=C_dx, C_rx=C_rx, C_px=C_px, D_out=D_out,
    d=256, n_heads=1, n_layers=1, dropout=0.5,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters : {n_params:,}")

# Verify init — embedding norms should be small (~0.01)
print(f"emb_dx weight std : {model.emb_dx.weight.data.std():.4f}  (expected ≈0.01)")
print(f"emb_rx weight std : {model.emb_rx.weight.data.std():.4f}")
print(f"emb_px weight std : {model.emb_px.weight.data.std():.4f}")

# Quick forward test
batch = next(iter(train_loader))
(dx_idx, dx_mask, dx_pad, dx_len,
 rx_idx, rx_mask, rx_pad, rx_len,
 px_idx, px_mask, px_pad, px_len,
 y, hist) = batch

logits_test = model(
    dx_idx.to(DEVICE), dx_mask.to(DEVICE),
    dx_pad.to(DEVICE), dx_len.to(DEVICE),
    rx_idx.to(DEVICE), rx_mask.to(DEVICE),
    rx_pad.to(DEVICE), rx_len.to(DEVICE),
    px_idx.to(DEVICE), px_mask.to(DEVICE),
    px_pad.to(DEVICE), px_len.to(DEVICE),
    hist.to(DEVICE),
)
print(f"Input  dx_idx  : {dx_idx.shape}   (B, T, L)")
print(f"Output logits  : {logits_test.shape}  → (B={y.shape[0]}, N={D_out})")
print(f"Logits range   : [{logits_test.min():.3f}, {logits_test.max():.3f}]  (should be small)")

/Users/r/Documents/cambridge_project/pyhealth-env/lib/python3.11/site-packages/pyhealth/tasks/cardiology_detect.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Model parameters : 3,064,636
emb_dx weight std : 0.0100  (expected ≈0.01)
emb_rx weight std : 0.0100
emb_px weight std : 0.0100
Input  dx_idx  : torch.Size([64, 92, 31])   (B, T, L)
Output logits  : torch.Size([64, 572])  → (B=64, N=572)
Logits range   : [-10000.000, 2.753]  (should be small)


## 7. Training

In [8]:
# Replace the loss function with ordinary multilabel BCE.
def masked_bce_loss(logits, y, hist_mask):
    return nn.functional.binary_cross_entropy_with_logits(logits, y)


def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    total_loss = torch.zeros((), device=DEVICE)
    n = 0
    val_logits, val_y = [], []

    with torch.set_grad_enabled(train):
        for batch in loader:
            (dx_idx, dx_mask, dx_pad, dx_len,
             rx_idx, rx_mask, rx_pad, rx_len,
             px_idx, px_mask, px_pad, px_len, y, hist) = batch

            dx_idx, dx_mask, dx_pad, dx_len = (
                dx_idx.to(DEVICE), dx_mask.to(DEVICE),
                dx_pad.to(DEVICE), dx_len.to(DEVICE)
            )
            rx_idx, rx_mask, rx_pad, rx_len = (
                rx_idx.to(DEVICE), rx_mask.to(DEVICE),
                rx_pad.to(DEVICE), rx_len.to(DEVICE)
            )
            px_idx, px_mask, px_pad, px_len = (
                px_idx.to(DEVICE), px_mask.to(DEVICE),
                px_pad.to(DEVICE), px_len.to(DEVICE)
            )
            y = y.to(DEVICE)

            logits = model(
                dx_idx, dx_mask, dx_pad, dx_len,
                rx_idx, rx_mask, rx_pad, rx_len,
                px_idx, px_mask, px_pad, px_len,
            )
            loss = nn.functional.binary_cross_entropy_with_logits(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            else:
                val_logits.append(logits.detach())
                val_y.append(y)

            total_loss += loss.detach() * y.size(0)
            n += y.size(0)

    if DEVICE.type == "mps":
        torch.mps.synchronize()

    mean_loss = (total_loss / n).item()

    if train:
        return mean_loss, None, None

    probs = torch.sigmoid(torch.cat(val_logits)).cpu().numpy()
    ys = torch.cat(val_y).cpu().numpy()
    return mean_loss, probs, ys

## 8. Evaluation

Primary metric: AUPRC (threshold-free, directly comparable to MedCo)
F1 reported at two thresholds:
  θ=0.5 (MedCo convention — fixed threshold)
  θ*    (optimised on validation, for diagnostic analysis)

In [9]:
def acc_at_m(probs, ys, m):
    order = np.argsort(-probs, axis=1)[:, :m]
    hits, denom = 0.0, 0.0
    for i in range(probs.shape[0]):
        true_idx = set(np.nonzero(ys[i])[0].tolist())
        if not true_idx:
            continue
        hits  += len(true_idx & set(order[i].tolist()))
        denom += len(true_idx)
    return hits / denom if denom > 0 else float('nan')


def stratified(probs, ys, threshold, cols, name):
    if not cols:
        return {}
    p, y = probs[:, cols], ys[:, cols]
    pred = (p > threshold).astype(int)
    aps  = [average_precision_score(y[:, j], p[:, j])
            for j in range(y.shape[1]) if y[:, j].sum() > 0]
    return {
        f'{name}_auprc':     float(np.mean(aps)) if aps else float('nan'),
        f'{name}_macro_f1':  f1_score(y, pred, average='macro', zero_division=0),
    }

def full_evaluation(probs, ys):
    # AUPRC — primary, threshold-free
    aps = [average_precision_score(ys[:, j], probs[:, j])
           for j in range(ys.shape[1]) if ys[:, j].sum() > 0]
    results = {'auprc': float(np.mean(aps)) if aps else float('nan')}

    # Stratified AUPRC by rarity — the key diagnostic metric
    for cols, name in [(rare_idx, 'rare'),
                       (medium_idx, 'medium'),
                       (common_idx, 'common')]:
        p, y = probs[:, cols], ys[:, cols]
        aps_s = [average_precision_score(y[:, j], p[:, j])
                 for j in range(y.shape[1]) if y[:, j].sum() > 0]
        results[f'auprc_{name}'] = float(np.mean(aps_s)) if aps_s else float('nan')

    # Acc@m — threshold-free ranking metric
    results[f'acc_at_{m_median}'] = acc_at_m(probs, ys, m_median)
    results[f'acc_at_{m_90}']     = acc_at_m(probs, ys, m_90)

    return results

## 9. Training Loop

In [10]:
import time

MAX_EPOCHS = 50
PATIENCE = 5
LR = 1e-3

def sample_auprc(probs, ys):
    """
    MedCo/PyHealth-style PR-AUC: average AP across patient samples.

    Rows with no positive labels are excluded because per-sample AP is
    undefined for an all-zero target.
    """
    keep = ys.sum(axis=1) > 0

    if keep.sum() == 0:
        return 0.0

    return float(
        average_precision_score(
            ys[keep],
            probs[keep],
            average="samples",
        )
    )


def macro_label_auprc(probs, ys):
    """
    Mean AP across diagnosis labels. Keep this for rare-label analysis;
    it is not the same metric as MedCo's headline sample-AUPRC.
    """
    aps = [
        average_precision_score(ys[:, j], probs[:, j])
        for j in range(ys.shape[1])
        if ys[:, j].sum() > 0
    ]
    return float(np.mean(aps)) if aps else 0.0


optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_auprc = -1.0
best_state = None
patience_ctr = 0
history = []

print(f"Training (max {MAX_EPOCHS} epochs, early stopping patience={PATIENCE})")
print("Early stopping criterion: validation sample-AUPRC")
print("-" * 70)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_loss, _, _           = run_epoch(model, train_loader, optimizer)
    val_loss, val_probs, val_y = run_epoch(model, val_loader)

    val_auprc             = sample_auprc(val_probs, val_y)
    val_macro_label_auprc = macro_label_auprc(val_probs, val_y)
    val_f1_05             = f1_score(
        val_y, (val_probs > 0.5).astype(int),
        average="macro", zero_division=0)

    elapsed = time.time() - t0

    history.append({
        "epoch":                 epoch,
        "train_loss":            train_loss,
        "val_loss":              val_loss,
        "val_sample_auprc":      val_auprc,
        "val_macro_label_auprc": val_macro_label_auprc,
        "val_macro_f1_t05":      val_f1_05,
        "epoch_time_s":          elapsed,
    })

    print(
        f"Epoch {epoch:>2} | {elapsed/60:.1f}min | "
        f"loss {train_loss:.4f} | "
        f"sample_AUPRC {val_auprc:.4f} | "
        f"macro_label_AUPRC {val_macro_label_auprc:.4f} | "
        f"F1@0.5 {val_f1_05:.4f}"
    )

    if epoch == 1:
        est_low  = elapsed / 60 * 10
        est_high = elapsed / 60 * 20
        print(f"\n  → Epoch 1: {elapsed/60:.1f} min "
              f"| estimated total: {est_low:.0f}–{est_high:.0f} min "
              f"({est_low/60:.1f}–{est_high/60:.1f} h)\n")

    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        best_state     = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        patience_ctr   = 0
        print(f"           ↑ new best sample-AUPRC: {val_auprc:.4f}")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch} "
                  f"(best sample-AUPRC={best_val_auprc:.4f})")
            break

model.load_state_dict(best_state)
checkpoint_path = CHECKPOINTS / "backbone_ccs.pt"
torch.save(model.state_dict(), checkpoint_path)
print(f"\nSaved checkpoint → {checkpoint_path}")

Training (max 50 epochs, early stopping patience=5)
Early stopping criterion: validation sample-AUPRC
----------------------------------------------------------------------
Epoch  1 | 33.3min | loss 0.0692 | sample_AUPRC 0.4477 | macro_label_AUPRC 0.1378 | F1@0.5 0.0605

  → Epoch 1: 33.3 min | estimated total: 333–666 min (5.5–11.1 h)

           ↑ new best sample-AUPRC: 0.4477
Epoch  2 | 91.5min | loss 0.0630 | sample_AUPRC 0.4618 | macro_label_AUPRC 0.1492 | F1@0.5 0.0745
           ↑ new best sample-AUPRC: 0.4618
Epoch  3 | 82.6min | loss 0.0621 | sample_AUPRC 0.4699 | macro_label_AUPRC 0.1596 | F1@0.5 0.0846
           ↑ new best sample-AUPRC: 0.4699
Epoch  4 | 61.9min | loss 0.0617 | sample_AUPRC 0.4680 | macro_label_AUPRC 0.1665 | F1@0.5 0.0819
Epoch  5 | 82.2min | loss 0.0614 | sample_AUPRC 0.4723 | macro_label_AUPRC 0.1707 | F1@0.5 0.0864
           ↑ new best sample-AUPRC: 0.4723
Epoch  6 | 147.9min | loss 0.0611 | sample_AUPRC 0.4703 | macro_label_AUPRC 0.1750 | F1@0.5 0.083

KeyboardInterrupt: 

In [13]:
# ============================================================
# MedCo-style TEST evaluation:
# sample-AUPRC overall + FOUR label-frequency quartiles
# ============================================================

import json
from collections import Counter
import numpy as np
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# 1. Restore the best validation model currently in memory.
#    This does NOT retrain or rebuild anything.
# ------------------------------------------------------------
model.load_state_dict(best_state)
model.eval()

# ------------------------------------------------------------
# 2. Rebuild MedCo-style frequency quartiles from TRAIN labels.
#
# Frequency = number of training samples in which a CCS label
# appears as a target. Test data are never used to define groups.
# ------------------------------------------------------------
train_label_counts = Counter()

for sample in train_samples:
    # `label` originates from a set in Notebook 1, but set() is
    # retained here defensively: one code counts once per sample.
    train_label_counts.update(set(sample["label"]))

vocab = meta["vocab"]  # same ordered label vocabulary as Notebook 1
train_freqs = np.array(
    [train_label_counts[code] for code in vocab],
    dtype=float,
)

q25, q50, q75 = np.percentile(train_freqs, [25, 50, 75])

q0_25_idx = [
    i for i, freq in enumerate(train_freqs)
    if freq <= q25
]
q25_50_idx = [
    i for i, freq in enumerate(train_freqs)
    if q25 < freq <= q50
]
q50_75_idx = [
    i for i, freq in enumerate(train_freqs)
    if q50 < freq <= q75
]
q75_100_idx = [
    i for i, freq in enumerate(train_freqs)
    if freq > q75
]

quartiles = {
    "0-25% (rarest)": q0_25_idx,
    "25-50%":          q25_50_idx,
    "50-75%":          q50_75_idx,
    "75-100% (common)": q75_100_idx,
}

print("Training-label frequency boundaries")
print(f"  Q25 = {q25:.0f}")
print(f"  Q50 = {q50:.0f}")
print(f"  Q75 = {q75:.0f}")
print()

for name, indices in quartiles.items():
    print(f"{name:18s}: {len(indices):3d} CCS codes")

# Safety check: every output label belongs to exactly one quartile.
all_quartile_indices = (
    q0_25_idx + q25_50_idx + q50_75_idx + q75_100_idx
)
assert len(all_quartile_indices) == D_out
assert len(set(all_quartile_indices)) == D_out

# ------------------------------------------------------------
# 3. MedCo / PyHealth sample-AUPRC functions
# ------------------------------------------------------------
def sample_auprc(probs, ys):
    """
    Overall sample-AUPRC:
    AP is computed within each patient/sample across diagnosis labels,
    then averaged across samples with >= 1 true diagnosis.
    """
    keep = ys.sum(axis=1) > 0

    if keep.sum() == 0:
        return float("nan")

    return float(
        average_precision_score(
            ys[keep],
            probs[keep],
            average="samples",
        )
    )


def quartile_sample_auprc(probs, ys, indices):
    """
    Sample-AUPRC restricted to one frequency quartile.

    Only test samples having at least one true label in that quartile
    are included. This matches the intended MedCo-style comparison
    across 0–25%, 25–50%, 50–75%, and 75–100% label groups.
    """
    group_probs = probs[:, indices]
    group_y = ys[:, indices]

    keep = group_y.sum(axis=1) > 0

    if keep.sum() == 0:
        return float("nan"), 0

    score = average_precision_score(
        group_y[keep],
        group_probs[keep],
        average="samples",
    )

    return float(score), int(keep.sum())

# ------------------------------------------------------------
# 4. One forward pass over the existing test loader
# ------------------------------------------------------------
_, test_probs, test_y = run_epoch(model, test_loader)

# ------------------------------------------------------------
# 5. Calculate overall and four-quartile sample-AUPRC
# ------------------------------------------------------------
overall_sample_auprc = sample_auprc(test_probs, test_y)

quartile_results = {}

for name, indices in quartiles.items():
    score, n_test_samples = quartile_sample_auprc(
        test_probs,
        test_y,
        indices,
    )

    quartile_results[name] = {
        "sample_auprc": score,
        "n_codes": len(indices),
        "n_test_samples_with_positive_label": n_test_samples,
    }

rare_common_gap = (
    quartile_results["75-100% (common)"]["sample_auprc"]
    - quartile_results["0-25% (rarest)"]["sample_auprc"]
)

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("BACKBONE TRANSFORMER — MEDCO-STYLE TEST SAMPLE-AUPRC")
print("=" * 72)
print(f"Overall sample-AUPRC : {overall_sample_auprc:.4f} "
      f"({100 * overall_sample_auprc:.2f}%)")
print("-" * 72)

for name, values in quartile_results.items():
    print(
        f"{name:18s} : "
        f"{values['sample_auprc']:.4f} "
        f"({100 * values['sample_auprc']:.2f}%)"
        f" | {values['n_codes']:3d} codes"
        f" | {values['n_test_samples_with_positive_label']:,} test samples"
    )

print("-" * 72)
print(f"75–100% minus 0–25% gap : {rare_common_gap:.4f} "
      f"({100 * rare_common_gap:.2f} percentage points)")
print("=" * 72)

# ------------------------------------------------------------
# 7. Save results and exact group definitions for reproducibility
# ------------------------------------------------------------
results_to_save = {
    "metric": "sample_AUPRC",
    "selection_checkpoint": "best_state",
    "frequency_source": "training target-label occurrences per sample",
    "quartile_boundaries": {
        "q25": float(q25),
        "q50": float(q50),
        "q75": float(q75),
    },
    "overall_sample_auprc": overall_sample_auprc,
    "quartiles": quartile_results,
    "common_minus_rarest_gap": rare_common_gap,
    "quartile_code_indices": {
        name: [int(i) for i in indices]
        for name, indices in quartiles.items()
    },
}

output_path = RESULTS / "backbone_ccs_medco_four_quartiles.json"

with open(output_path, "w") as f:
    json.dump(results_to_save, f, indent=2)

print(f"\nSaved → {output_path}")

Training-label frequency boundaries
  Q25 = 200
  Q50 = 1132
  Q75 = 4258

0-25% (rarest)    : 143 CCS codes
25-50%            : 143 CCS codes
50-75%            : 143 CCS codes
75-100% (common)  : 143 CCS codes

BACKBONE TRANSFORMER — MEDCO-STYLE TEST SAMPLE-AUPRC
Overall sample-AUPRC : 0.4707 (47.07%)
------------------------------------------------------------------------
0-25% (rarest)     : 0.2896 (28.96%) | 143 codes | 702 test samples
25-50%             : 0.3052 (30.52%) | 143 codes | 4,915 test samples
50-75%             : 0.3674 (36.74%) | 143 codes | 10,667 test samples
75-100% (common)   : 0.5429 (54.29%) | 143 codes | 14,812 test samples
------------------------------------------------------------------------
75–100% minus 0–25% gap : 0.2533 (25.33 percentage points)

Saved → ../results/backbone_ccs_medco_four_quartiles.json


In [ ]:
# ============================================================
# Cell 9b — Training curves
# ============================================================
import matplotlib.pyplot as plt

epochs      = [h['epoch']                 for h in history]
train_loss  = [h['train_loss']            for h in history]
val_loss    = [h['val_loss']              for h in history]
sample_auc  = [h['val_sample_auprc']      for h in history]
macro_auc   = [h['val_macro_label_auprc'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: losses
axes[0].plot(epochs, train_loss, label='train loss', color='steelblue')
axes[0].plot(epochs, val_loss,   label='val loss',   color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Train vs Val Loss')
axes[0].legend()

# Right: AUPRC metrics
axes[1].plot(epochs, sample_auc, label='sample AUPRC', color='steelblue')
axes[1].plot(epochs, macro_auc,  label='macro label AUPRC', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUPRC')
axes[1].set_title('Validation AUPRC over training')
axes[1].legend()

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'training_curves.png', dpi=150)
plt.show()

print(f"Best sample AUPRC      : {max(sample_auc):.4f} (epoch {epochs[sample_auc.index(max(sample_auc))]})")
print(f"Best macro label AUPRC : {max(macro_auc):.4f} (epoch {epochs[macro_auc.index(max(macro_auc))]})")

## 10. Test Evaluation

In [ ]:
_, test_probs, test_y = run_epoch(model, test_loader)
results = full_evaluation(test_probs, test_y)

print("\n" + "=" * 65)
print("BACKBONE TRANSFORMER — TEST RESULTS (CCS label space)")
print("=" * 65)
print(f"{'AUPRC (primary)':>30s} : {results['auprc']:.4f}")
print(f"  {'→ MedCo baseline reports':>28s} : ~46.80")
print("-" * 65)
print(f"{'AUPRC  rare':>30s} : {results.get('auprc_rare',   float('nan')):.4f}")
print(f"{'AUPRC  medium':>30s} : {results.get('auprc_medium', float('nan')):.4f}")
print(f"{'AUPRC  common':>30s} : {results.get('auprc_common', float('nan')):.4f}")
gap = results.get('auprc_common', 0) - results.get('auprc_rare', 0)
print(f"{'rare/common AUPRC gap':>30s} : {gap:.4f}")
print(f"  {'→ MedCo baseline gap':>28s} : ~29.18")
print("-" * 65)
print(f"{'Acc @ median':>30s} : {results[f'acc_at_{m_median}']:.4f}")
print(f"{'Acc @ 90th pct':>30s} : {results[f'acc_at_{m_90}']:.4f}")
print("=" * 65)

with open(RESULTS / 'backbone_ccs_results.json', 'w') as f:
    json.dump({
        'model': 'SharedBackbone',
        'label_space': 'CCS',
        'architecture': {
            'd': 256, 'n_heads': 1, 'n_layers': 1,
            'dropout': 0.5, 'pooling': 'sum',
            'transformers': 'separate_per_modality',
        },
        'test': {k: float(v) for k, v in results.items()},
        'history': history,
    }, f, indent=2)

print("Saved → results/backbone_ccs_results.json")